In [1]:
import os
import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import yfinance as yf

from datetime import datetime
from typing import List

In [2]:
def market_close_asof(date_local: pd.Timestamp, cal_code: str = "XNYS") -> List[pd.Timestamp]:
    """
    Return the most recent market-close timestamp in UTC following a given market calendar.

    Args:
        date_local (pd.Timestamp):  Find the most recent market-close as of this local timestamp.
        cal_code (str):             Code corresponding to market calendar, e.g. XNYS (NYSE), XLON, XETR.
    
    Returns:
        List[pd.Timestamp] constianing the dates of the two most recent market closes (tz-aware, UTC).
    """
    
    cal = mcal.get_calendar(cal_code)

    # Build a small date window around date of interest to locate prev/next sessions
    date_utc = date_local.tz_convert("UTC")
    start = (date_utc - pd.Timedelta(days = 10)).date()
    end = (date_utc + pd.Timedelta(days = 10)).date()

    # Full schedule with market_open/market_close in calendar's local timezone
    sched = cal.schedule(start_date = start, end_date = end)

    # Convert schedule times to UTC for robust comparisons
    sched_utc = sched.copy()
    sched_utc["market_open_utc"] = sched_utc["market_open"].dt.tz_convert("UTC")
    sched_utc["market_close_utc"] = sched_utc["market_close"].dt.tz_convert("UTC")
    past_closes = sched_utc[sched_utc["market_close_utc"] <= date_utc]["market_close_utc"]

    # Is the market open?
    date_str = date_utc.date().isoformat()
    date_row = sched_utc.loc[date_str] if date_str in sched_utc.index else None

    # Determine the most recent completed close
    if date_row is not None:
        use_today = (date_utc >= date_row["market_close_utc"])
        if use_today:
            most_recent_close = date_row["market_close_utc"]
            prev_close = past_closes.iloc[-2]
        else:
            most_recent_close = past_closes.iloc[-1]
            prev_close = past_closes.iloc[-2]
    else:
        most_recent_close = past_closes.iloc[-1]
        prev_close = past_closes.iloc[-2]
        
    return [most_recent_close, prev_close]

def continuous_rate_from_annual_percent(pct_yield: float) -> float:
    """Convert a quoted annualized percent yield to continuous compounding."""
    y = float(pct_yield) / 100.0
    return np.log(1.0 + max(y, 0.0))

def exp_option_chain(ticker: str, exp_chain: yf.ticker.Ticker, exp: str, df_rows: List[pd.DataFrame], 
                     as_of_date: pd.Timestamp, S: float, r: float, q: float) -> List[pd.DataFrame]:

    """
    Aggregate option chain for given ticker at specific expiration.

    Args:
        ticker (str):                   Stock ticker symbol.
        exp_chain (yf.ticker.Ticker):   Option chain object with two DataFrames, one containing all the call options data 
                                        for the selected expiration date and the other all the put options data.
        exp (str):                      Option expiration date as 'YYYY-MM-DD'.
        df_rows (List[pd.DataFrame]):   List of option chain DataFrames.
        as_of_date (pd.Timestamp):      Today's date.
        S (float):                      Spot price.
        r (float):                      Risk-free rate.
        q (float):                      Dividend yield.

    Returns:
        List[pd.DataFrame] of option chain DataFrames with those corresponding to the expiration date of interest appended.
    """
    for option_type, option_df in [("call", exp_chain.calls), ("put", exp_chain.puts)]:
        
        if option_df is None or option_df.empty:
            continue
        
        tmp = option_df.copy()
        tmp["type"] = option_type
        tmp["symbol"] = ticker
        tmp["expiry"] = pd.to_datetime(exp, utc=True)
        tmp["date"] = as_of_date
        tmp["underlying"] = S
        tmp["rate"] = r
        tmp["div_yield"] = q

        # Parse lastTradeDate and compute quote age (hours)
        if "lastTradeDate" in tmp.columns:
            tmp["lastTradeDate"] = pd.to_datetime(tmp["lastTradeDate"], utc=True, errors="coerce")
            tmp["quote_age_hours"] = (tmp["date"] - tmp["lastTradeDate"]).dt.total_seconds().clip(lower = 0) / 3600.0
        else:
            tmp["lastTradeDate"] = pd.NaT
            tmp["quote_age_hours"] = np.nan

        tmp.rename(columns={"impliedVolatility": "iv_mkt",
                            "lastPrice": "last"}, inplace=True)
        cols = ["date", "expiry", "symbol", "contractSymbol", "type", "inTheMoney", "strike", "bid", "ask", "last", 
                "underlying", "iv_mkt", "rate", "div_yield", "volume", "openInterest", "lastTradeDate", "quote_age_hours"]
        tmp = tmp[[c for c in cols if c in tmp.columns]]
        df_rows.append(tmp)
    
    return df_rows

def fetch_option_chains(ticker: str, market_cal: str, as_of: str | None = None) -> pd.DataFrame: 
    """
    Fetch raw option chains (calls & puts) for all expirations for the given ticker from yfinance.

    Args:
        ticker (str):       Stock ticker symbol.
        market_cal (str):   Code corresponding to market calendar, e.g. XNYS (NYSE), XNAS (NASDAQ), XLON, XETR.
        as_of (str | None): Date as 'YYYY-MM-DD'. 

    Returns:
        pd.DataFrame containing option chains data for all expirations and option types.
    """
    # Market-close timestamp
    if as_of is None:
        # Most recent market-close as of today
        now_utc = pd.Timestamp.now(tz = "UTC")
        [as_of, prev_close] = market_close_asof(now_utc, market_cal)
    else:
        # Market-close on given date
        date_utc = pd.Timestamp(as_of).tz_localize("UTC")
        date_utc = date_utc.normalize() + pd.Timedelta(hours = 23, minutes = 59)
        [as_of, prev_close] = market_close_asof(date_utc, market_cal)
    
    # Create ticker object
    tk = yf.Ticker(ticker)

    # Available option expiry dates
    expiries = tk.options
    if len(expiries) == 0:
        raise RuntimeError(f"No option expiries available from yfinance for {ticker} right now.")

    # Pull 1 day of price history
    hist = tk.history(start = prev_close, end = as_of)
    if hist.empty:
        raise RuntimeError(f"No price history returned for {ticker}.")
    
    # Use latest close as spot
    S0 = float(hist["Close"].iloc[-1])

    # Risk-Free Proxy: 13-Week T-Bill
    try:
        irx = yf.Ticker("^IRX").history(period="5d")["Close"].dropna().iloc[-1]
        r_cont = continuous_rate_from_annual_percent(irx)
    except Exception:
        r_cont = 0.02 # fallback
    
    # Dividend Yield Proxy: trailing 365-day dividends / spot
    try:
        divs = tk.dividends
        recent = divs[divs.index >= (as_of - pd.Timedelta(days=365))]
        # Sum the last 365 days of dividends and divide by spot
        q_cont = float(recent.sum() / S0) if not recent.empty else 0.0 # NOT continuous q but proxy for
    except Exception:
        q_cont = 0.0 # fallback
    
    # Aggregate option chains
    rows = []
    for exp in expiries:
        chain = tk.option_chain(exp)
        rows = exp_option_chain(ticker, chain, exp, rows, as_of, S0, r_cont, q_cont)    
    df = pd.concat(rows, ignore_index=True)

    # Compute mid price from bid/ask when both exist; otherwise fall back to last price
    df["mid"] = np.where(np.isfinite(df[["bid", "ask"]]).all(axis = 1), 0.5 * (df["bid"] + df["ask"]), df["last"])
    
    # Compute time to expiry in years using Actual/365
    df["T"] = (pd.to_datetime(df["expiry"]) - pd.to_datetime(df["date"])).dt.days.clip(lower = 0).astype(float) / 365.0

    return df

def save_raw_data(df: pd.DataFrame, ticker: str, outdir: str, run_date: str):
    """
    Save the option chains dataframe as CSV:
        {outdir}/{ticker}_{run_date}.csv
    
    Args:
        df (pd.DataFrame):      Raw option chains data.
        ticker (str):           Ticker for underlying.
        outdir (str):           Path to data directory.
        run_date (str):         Today's date as "YYYYMMDD".
    """
    filename = f"{ticker}_{run_date}.csv"
    path = os.path.join(outdir, filename)
    df.to_csv(path, index = False)
    print(f"Saved {run_date} {ticker} option chains to {path}")

# Single Ticker

In [5]:
# Inputs
TICKER      = "^VIX"
MARKET_CAL  = "XNYS"
OUTDIR      = "data/options"
AS_OF       = datetime.now().strftime("%Y%m%d") 

In [6]:
rootdir_path = os.path.abspath(os.path.join("..")) # project root
outdir_path = rootdir_path + "/" + OUTDIR
if not os.path.exists(outdir_path):
    os.makedirs(outdir_path)

option_chains_df  = fetch_option_chains(TICKER, MARKET_CAL, AS_OF)
save_raw_data(option_chains_df, TICKER, outdir_path, AS_OF)

Saved 20251118 ^VIX option chains to /Users/oanamirestean/Documents/optionsvolstrat/data/options/^VIX_20251118.csv


# Multiple Tickers

In [3]:
# Inputs
TICKERS     = ["ADBE", "AMD", "AMZN", "ANET", "AVGO", "GOOGL", "GTLB", "IBM", "META", "MSFT", "NET", 
               "NOW", "NVDA", "ORCL", "PLTR", "QQQ", "RING", "SMH", "SPY", "^VIX", "VOO", "VT"]
MARKET_CAL  = "XNYS"
OUTDIR      = "data/options"
AS_OF       = datetime.now().strftime("%Y%m%d") 

In [4]:
rootdir_path = os.path.abspath(os.path.join("..")) # project root
outdir_path = rootdir_path + "/" + OUTDIR
if not os.path.exists(outdir_path):
    os.makedirs(outdir_path)

for TICKER in TICKERS:
    option_chains_df  = fetch_option_chains(TICKER, MARKET_CAL, AS_OF)
    save_raw_data(option_chains_df, TICKER, outdir_path, AS_OF)

Saved 20251121 ADBE option chains to /Users/oanamirestean/Documents/optionsvolstrat/data/options/ADBE_20251121.csv
Saved 20251121 AMD option chains to /Users/oanamirestean/Documents/optionsvolstrat/data/options/AMD_20251121.csv
Saved 20251121 AMZN option chains to /Users/oanamirestean/Documents/optionsvolstrat/data/options/AMZN_20251121.csv
Saved 20251121 ANET option chains to /Users/oanamirestean/Documents/optionsvolstrat/data/options/ANET_20251121.csv
Saved 20251121 AVGO option chains to /Users/oanamirestean/Documents/optionsvolstrat/data/options/AVGO_20251121.csv
Saved 20251121 GOOGL option chains to /Users/oanamirestean/Documents/optionsvolstrat/data/options/GOOGL_20251121.csv
Saved 20251121 GTLB option chains to /Users/oanamirestean/Documents/optionsvolstrat/data/options/GTLB_20251121.csv
Saved 20251121 IBM option chains to /Users/oanamirestean/Documents/optionsvolstrat/data/options/IBM_20251121.csv
Saved 20251121 META option chains to /Users/oanamirestean/Documents/optionsvolstra